# Checkpoint Three: Cleaning Data

Now you are ready to clean your data. Before starting coding, provide the link to your dataset below.

My dataset: [Link to original data](https://www.kaggle.com/datasets/open-powerlifting/powerlifting-database?resource=download&select=openpowerlifting.csv)

Import the necessary libraries and create your dataframe(s).

In [1]:
# Import core libraries for data manipulation
import pandas as pd

# Load the dataset CSV into a pandas DataFrame
df = pd.read_csv("openpowerlifting.csv", dtype=str)
# Quick preview to confirm the data loaded correctly
df.head()


,Name,Sex,Event,Equipment,Age,AgeClass,Division,BodyweightKg,WeightClassKg,Squat1Kg,...,McCulloch,Glossbrenner,IPFPoints,Tested,Country,Federation,Date,MeetCountry,MeetState,MeetName
0,Abbie Murphy,F,SBD,Wraps,29,24-34,F-OR,59.8,60,80,...,324.16,286.42,511.15,NaN,NaN,GPC-AUS,2018-10-27,Australia,VIC,Melbourne Cup
1,Abbie Tuong,F,SBD,Wraps,29,24-34,F-OR,58.5,60,100,...,378.07,334.16,595.65,NaN,NaN,GPC-AUS,2018-10-27,Australia,VIC,Melbourne Cup
2,Ainslee Hooper,F,B,Raw,40,40-44,F-OR,55.4,56,NaN,...,38.56,34.12,313.97,NaN,NaN,GPC-AUS,2018-10-27,Australia,VIC,Melbourne Cup
3,Amy Moldenhauer,F,SBD,Wraps,23,20-23,F-OR,60,60,-105,...,345.61,305.37,547.04,NaN,NaN,GPC-AUS,2018-10-27,Australia,VIC,Melbourne Cup
4,Andrea Rowan,F,SBD,Wraps,45,45-49,F-OR,104,110,120,...,338.91,274.56,550.08,NaN,NaN,GPC-AUS,2018-10-27,Australia,VIC,Melbourne Cup


## Missing Data

Test your dataset for missing data and handle it as needed. Make notes in the form of code comments as to your thought process.

In [10]:
# Check for missing values in all columns
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts.head(20)  # Look at the 20 columns with the most missing values


# Since our analysis focuses on bench press, sex, and country, we need these columns to be complete
# Drop rows where 'Best3BenchKg', 'Sex', or 'Country' is missing
df_clean = df.dropna(subset=["Best3BenchKg", "Sex", "Country"])

# Confirm that the missing values in those columns are gone
df_clean[["Best3BenchKg", "Sex", "Country"]].isna().sum()


# Note: Other columns may still have missing values (e.g., Age, BodyweightKg) but they are not critical to our analysis


Best3BenchKg    0
Sex             0
Country         0
dtype: int64

## Irregular Data

Detect outliers in your dataset and handle them as needed. Use code comments to make notes about your thought process.

In [11]:
# First, check for negative bench press values, these values are not valid because a bench press cannot be negative
negative_bench = df_clean[df_clean["Best3BenchKg"] < 0]
print("Number of negative bench press values:", negative_bench.shape[0])

# Remove rows with negative bench press values
df_clean = df_clean[df_clean["Best3BenchKg"] >= 0]

# Confirm removal
print("Rows after removing negative bench press values:", df_clean.shape[0])


# Next, check for extremely high bench press values
# I will look at the summary statistics to see if there are unrealistic outliers
df_clean["Best3BenchKg"].describe()


# Note: For this project, I will keep very high values because they are possible for elite lifters.
# Only obviously invalid values (like negatives) are removed.


Number of negative bench press values: 516
Rows after removing negative bench press values: 343820


count    343820.000000
mean        143.901895
std          59.546834
min           7.500000
25%          97.500000
50%         140.000000
75%         182.500000
max         488.500000
Name: Best3BenchKg, dtype: float64

## Unnecessary Data

Look for the different types of unnecessary data in your dataset and address it as needed. Make sure to use code comments to illustrate your thought process.

In [ ]:
# Filter to U.S. lifters BEFORE removing columns, we are only looking at USA lifters for this analysis
df_us = df_clean[df_clean["Country"] == "USA"]  # or "Country" if that column exists in your raw df

# Only keeping coumns I may need for my analysis
columns_to_keep = ["Name", "Sex", "Best3BenchKg", "BodyweightKg", "Age", "AgeClass"]
df_us = df_us[columns_to_keep]

# Step 3: Reset index
df_us.reset_index(drop=True, inplace=True)

# Preview the cleaned dataframe
df_us.head()


,Name,Sex,Best3BenchKg,BodyweightKg,Age,AgeClass
0,Brett Worland,M,195.0,98.1,23.0,20-23
1,Michael Hamilton,M,95.0,69.9,19.0,18-19
2,Dan Green,M,235.0,109.6,30.0,24-34
3,Brett Worland,M,215.5,99.8,27.0,24-34
4,Matthew Webb,M,216.0,109.1,33.0,24-34


## Inconsistent Data

Check for inconsistent data and address any that arises. As always, use code comments to illustrate your thought process.

In [14]:
# Check unique values in 'Sex' to make sure it's consistent we only want 'M' and 'F'. If there are other values like 'm', 'f', or blanks, we need to fix them.
df_clean["Sex"].unique()


# Standardize capitalization (if needed)
df_clean["Sex"] = df_clean["Sex"].str.upper()

# Confirm changes
df_clean["Sex"].unique()


# Check for inconsistent numeric values
# Bodyweight should be > 0
df_clean = df_clean[df_clean["BodyweightKg"] > 0]

# Confirm numeric column clean-up
df_clean["BodyweightKg"].describe()


# Check for duplicate names at the same weight and bench since this might mean duplicate records
duplicates = df_clean.duplicated(subset=["Name", "BodyweightKg", "Best3BenchKg"])
print("Number of potential duplicates:", duplicates.sum())


Number of potential duplicates: 27835


## Summarize Your Results

Make note of your answers to the following questions.

1. Did you find all four types of dirty data in your dataset?
   - **Missing data:** Yes, there were rows with missing bench press, sex, or country information.  
   - **Irregular data:** Yes, there were negative bench press values.  
   - **Unnecessary data:** Yes, many columns like Squat, Deadlift, and MeetName were not needed for this analysis.  
   - **Inconsistent data:** Yes, there were capitalization differences in 'Sex' and minor typos in 'Equipment'.  
2. Did the process of cleaning your data give you new insights into your dataset?
   - Yes, I noticed that the dataset has far more men than women.  
   - I also realized that equipment type can vary a lot and may be something to take into account.  
   - Removing invalid or missing data helped me understand which parts of the dataset are reliable for answering my project question.
3. Is there anything you would like to make note of when it comes to manipulating the data and making visualizations?
   - Filtering rows should be done before dropping columns. 
   - Standardizing categorical data (like 'Sex' and 'Equipment') makes group comparisons and bar charts more accurate.  
   - Some extreme values are valid and should not always be removed.  
   - Keeping the dataset clean and focused makes it much easier to create clear visualizations later.
